# 🚢 Notebook 1: The Bulkhead Pattern — Intro

### The ship analogy

Big ships are built with **watertight bulkheads** — internal walls that split the hull into sealed compartments. If one compartment springs a leak, water stays there. The other compartments stay dry and the ship stays afloat.

### In software

A running service has **shared resources**: threads, database connections, HTTP connections, memory. If a single slow or broken downstream dependency hogs all of them, every other piece of your app gets starved too — even the healthy bits.

The **Bulkhead pattern** partitions those shared resources into **separate pools, one per dependency (or per tenant, or per workload class)** so that one failure stays contained.

> **One-line definition:** *Give each dependency its own small, bounded pool of resources so a problem in one can't drain the whole system.*


## 🛠️ Setup

```bash
cd 05-microservices/bulkhead
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

This notebook uses only the Python standard library (`concurrent.futures`, `threading`, `time`) — no extra installs needed.


## 🧪 The problem: one shared pool is a single point of failure

Imagine a web service that calls **two** downstream services:

- `users` — normally fast (50 ms)
- `reports` — normally fast, but today it's sick and takes 2 s per call

Both calls go through the **same** thread pool. Let's see what happens.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

# Simulated downstream services
def users_service():
    time.sleep(0.05)
    return "user-data"

def reports_service_sick():
    # Today reports is slow — each call takes 2 seconds
    time.sleep(2.0)
    return "report-data"

# ONE shared pool for BOTH dependencies. This is the "bad" version.
shared_pool = ThreadPoolExecutor(max_workers=4)

# A burst of 4 slow 'reports' calls arrives and fills the pool
slow_futures = [shared_pool.submit(reports_service_sick) for _ in range(4)]

# Now a healthy 'users' call comes in...
t0 = time.time()
users_future = shared_pool.submit(users_service)
users_future.result()   # wait for it
elapsed = time.time() - t0

print(f"users call waited {elapsed:.2f}s before it could even start")
print("the 'reports' problem has infected 'users' - classic resource starvation")

shared_pool.shutdown(wait=True)


Even though `users` itself is healthy and fast, it was **blocked by reports** because they share the same thread pool. This is how a single slow dependency brings down an entire service. The failure *propagated*.

## 🛡️ The fix: give each dependency its own bulkhead

We split the single pool into **two bounded pools**, one per downstream. Now a flood in one compartment can't reach the other.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# Two small, bounded pools - these are our bulkheads.
users_pool   = ThreadPoolExecutor(max_workers=2, thread_name_prefix="users")
reports_pool = ThreadPoolExecutor(max_workers=2, thread_name_prefix="reports")

# Flood the reports bulkhead with slow work (simulating the outage)
for _ in range(8):
    reports_pool.submit(reports_service_sick)

# A healthy 'users' call arrives at the SAME moment
t0 = time.time()
result = users_pool.submit(users_service).result()
elapsed = time.time() - t0

print(f"users call finished in {elapsed:.2f}s - healthy even though reports is drowning")
print(f"returned: {result}")

users_pool.shutdown(wait=False)
reports_pool.shutdown(wait=False)


### What changed?
- `users_pool` and `reports_pool` each have **their own threads**.
- When `reports` gets slow, it only exhausts its own 2 threads. The `users_pool` is untouched.
- Failure was **contained** — exactly like a watertight compartment.

## 🧵 Another flavor: semaphore-based bulkheads

A thread pool isn't the only way. You can also limit concurrency with a **semaphore** — a counter that says *"at most N callers may be inside this section at a time."* This is useful when the calls themselves are already async/non-blocking and you just want to cap how many are in flight.

In [ ]:
import threading, time

class SemaphoreBulkhead:
    """Limits how many concurrent callers can enter a protected section."""
    def __init__(self, max_concurrent: int, name: str):
        self.sem = threading.BoundedSemaphore(max_concurrent)
        self.name = name

    def call(self, fn, *args, **kwargs):
        # Try to enter. If full, refuse fast instead of queueing forever.
        acquired = self.sem.acquire(timeout=0.1)
        if not acquired:
            raise RuntimeError(f"[{self.name}] bulkhead FULL - rejecting call")
        try:
            return fn(*args, **kwargs)
        finally:
            self.sem.release()

# Protect the 'reports' dependency with a bulkhead of size 2
reports_bulkhead = SemaphoreBulkhead(max_concurrent=2, name="reports")

def try_call(i):
    try:
        reports_bulkhead.call(reports_service_sick)
        print(f"call {i}: ok")
    except RuntimeError as e:
        print(f"call {i}: {e}")

# Launch 5 callers at once - only 2 should get in
threads = [threading.Thread(target=try_call, args=(i,)) for i in range(5)]
for t in threads: t.start()
for t in threads: t.join()


Notice that the 3 extra callers were **rejected quickly** rather than piling up in a queue. This is called **shedding load** — it's a feature, not a bug. Telling callers *"no"* fast is much healthier than silently queuing them for 30 seconds.

## 📚 Beyond threads: where else bulkheads apply

The same idea shows up everywhere resources are shared:

| Resource | How to bulkhead it |
|---|---|
| **Database connections** | Give each service/tenant its own connection pool |
| **HTTP client sockets** | Separate `httpx.AsyncClient` / `requests.Session` per dependency |
| **Message consumers** | Dedicated consumer groups per tenant or workload type |
| **CPU / memory** | Run dependencies in separate processes, containers, or k8s pods |
| **Disk I/O** | Separate volumes or quotas per noisy neighbor |

> 💡 Rule of thumb: *every* boundary between you and an external dependency is a candidate for a bulkhead.

## 🎯 Takeaways

- **Bulkheads = resource partitioning.** One dependency's trouble stays in its own compartment.
- **Each pool must be bounded.** An unbounded pool is not a bulkhead — it's just a delayed disaster.
- **Reject fast when a bulkhead is full.** Queuing forever just shifts the crash to another layer.
- Bulkheads pair naturally with **timeouts**, **retries**, and **circuit breakers** — each solves a different failure mode.

Next up: a realistic worked example with an Order Service calling Inventory and Shipping. 👉 `02_worked_example.ipynb`
